# 🧠 Neural Network Playground

**Time:** 45–60 minutes  
**Goal:** See how a neural network learns, makes mistakes, and changes as we make it bigger or smaller — all without writing any code!

## What is a Neural Network?

Think of a neural network like a brain made of math! Just like you learn to recognize your friend's face by seeing it many times, a neural network learns to recognize patterns by looking at lots of examples.

## What We'll Do Today

We'll run **3 fun experiments** to see how neural networks work:

1. **Teach the Computer to Read Numbers** — Watch a computer learn to recognize handwritten digits (like 0, 1, 2, 3...)
2. **Too Smart for Its Own Good** — See what happens when a computer "memorizes" too much
3. **Draw the Decision Boundary** — Watch how a computer draws lines to separate different groups

## How to Use This Notebook

- Just click the **▶️ Run** button on each cell (or press Shift+Enter)
- You can also click **Run All** in the menu to run everything at once
- **You don't need to change any code!** Just follow along and use the sliders and dropdowns to explore.


## 📦 Step 1: Loading the Tools We Need

Before we can start, we need to load some special tools (called "libraries") that help us work with neural networks. Think of these like tools in a toolbox — we need the right tools to build and train our neural network.

**What each tool does:**
- `numpy` — Helps us work with numbers and arrays (lists of numbers)
- `matplotlib` — Helps us create graphs and pictures to visualize what's happening
- `torch` (PyTorch) — This is the main tool for building and training neural networks
- `sklearn` — Helps us create test problems and split our data
- `ipywidgets` — Creates the sliders and dropdowns you'll use to interact with the experiments


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.figsize'] = (10, 6)


---

## 🧩 Part 1: Teach the Computer to Read Numbers

### What's Happening Here?

Imagine you're learning to read someone's handwriting. At first, you might not be able to tell if a squiggle is a "5" or a "6". But after seeing lots of examples, you get better!

That's exactly what we're going to do with our computer. We'll show it thousands of pictures of handwritten numbers (0 through 9), and it will learn to recognize them.

### How It Works

1. **Training:** The computer looks at thousands of number pictures and their correct answers
2. **Learning:** It slowly gets better at guessing the right number
3. **Testing:** We show it new pictures it's never seen before to see how well it learned


### Step 1.1: Getting the Data

First, we need to get our training data. We'll use something called the **MNIST dataset** — this is a famous collection of 70,000 handwritten digit images that researchers use to test neural networks.

**What's happening in the code below:**
- We're downloading thousands of images of handwritten numbers (0-9)
- Each image is 28 pixels by 28 pixels (like a tiny black and white photo)
- We split the data into two groups: training data (to learn from) and test data (to test how well it learned)
- We also save 100 test images so you can try them out yourself later!


In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)


Now we'll organize the data so the computer can learn from it efficiently. We create something called "data loaders" that feed the images to the computer in small groups (called "batches"). This is like studying flashcards in small groups instead of all at once — it helps the computer learn better!


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

test_images = []
test_labels = []
for i in range(100):
    test_images.append(test_dataset[i][0])
    test_labels.append(test_dataset[i][1])
test_images = torch.stack(test_images)
test_labels = torch.tensor(test_labels)


### Step 1.2: Building the Neural Network Structure

Now we need to build the "brain" of our neural network. Think of this like building a house — we need to decide how many rooms (layers) and how big each room should be (how many neurons).

**Understanding the structure:**

1. **Input Layer:** Takes in the image (28×28 = 784 numbers total)
2. **Hidden Layer:** This is where the "thinking" happens. We have 64 neurons here — think of each neuron as a tiny decision-maker that looks for patterns
3. **Output Layer:** Gives us 10 answers — one for each possible digit (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)

**What is ReLU?**

ReLU stands for "Rectified Linear Unit" — but you don't need to remember that! Here's what it does in simple terms:

Think of ReLU like a light switch. If a neuron gets a positive signal, it "turns on" and passes that signal forward. If it gets a negative signal, it "turns off" and passes zero. This helps the network learn which patterns are important and which to ignore.

**What is Softmax?**

The output layer uses something called "softmax" (we'll see this in the training code). This takes all 10 possible answers and converts them into probabilities (percentages). So instead of just saying "I think it's a 5", it says "I'm 85% sure it's a 5, 10% sure it's a 6, and 5% sure it's something else". This gives us confidence scores!


### Step 1.2: Building the Neural Network Structure

Now we need to build the "brain" of our neural network. Think of this like building a house — we need to decide how many rooms (layers) and how big each room should be (how many neurons).

**Understanding the structure:**

1. **Input Layer:** Takes in the image (28×28 = 784 numbers total)
2. **Hidden Layer:** This is where the "thinking" happens. We have 64 neurons here — think of each neuron as a tiny decision-maker that looks for patterns
3. **Output Layer:** Gives us 10 answers — one for each possible digit (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)

**What is ReLU?**

ReLU stands for "Rectified Linear Unit" — but you don't need to remember that! Here's what it does in simple terms:

Think of ReLU like a light switch. If a neuron gets a positive signal, it "turns on" and passes that signal forward. If it gets a negative signal, it "turns off" and passes zero. This helps the network learn which patterns are important and which to ignore.

**What is Softmax?**

The output layer uses something called "softmax" (we'll see this in the training code). This takes all 10 possible answers and converts them into probabilities (percentages). So instead of just saying "I think it's a 5", it says "I'm 85% sure it's a 5, 10% sure it's a 6, and 5% sure it's something else". This gives us confidence scores!


In [ ]:
class NumberReader(nn.Module):
    def __init__(self):
        super(NumberReader, self).__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(28 * 28, 64)
        self.layer2 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.layer1(x))
        x = self.layer2(x)
        return x


Now we create an instance of our neural network (like building one house from our blueprint):


In [ ]:
model1 = NumberReader()


### Step 1.3: Setting Up How the Network Will Learn

Before training, we need to tell the computer HOW to learn. This is like setting up the rules for a game:

**Optimizer (Adam):** This is like a smart teacher that helps the network learn. It figures out how much to change the network's "brain" each time it makes a mistake. Adam is a popular choice because it's good at finding the right balance — not changing too much (which could make things worse) or too little (which would make learning too slow).

**Loss Function (Cross Entropy Loss):** This measures how wrong the network's guess was. Think of it like a test score — if the network guessed "5" but the answer was "3", the loss function calculates how big that mistake was. The network's goal is to make this number as small as possible!


In [ ]:
optimizer = optim.Adam(model1.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


### Step 1.4: Training the Model

Now comes the exciting part — actually training the network! This is where the computer learns to recognize numbers.

**What happens during training:**

1. The network looks at a batch of images and makes guesses
2. It compares its guesses to the correct answers
3. It calculates how wrong it was (using the loss function)
4. It adjusts its "brain" slightly to do better next time (using the optimizer)
5. This repeats many times until the network gets good at recognizing numbers

**What is an Epoch?**

An "epoch" is one complete pass through all the training data. We'll train for 5 epochs, which means the network will see every training image 5 times. Each time it sees the images, it gets a little bit better!

**Training vs. Test Accuracy:**

- **Training Accuracy:** How well the network does on the images it's learning from
- **Test Accuracy:** How well it does on brand new images it's never seen before

We want both to be high! If training accuracy is much higher than test accuracy, it means the network might be "memorizing" instead of truly learning (we'll explore this more in Part 2).


In [ ]:
train_accuracies = []
val_accuracies = []

for epoch in range(5):
    model1.train()
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model1(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    train_acc = 100 * correct_train / total_train
    train_accuracies.append(train_acc)
    
    model1.eval()
    correct_test = 0
    total_test = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model1(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()
    
    val_acc = 100 * correct_test / total_test
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch + 1}/5: Training = {train_acc:.1f}%, Test = {val_acc:.1f}%")


### Step 1.5: Visualizing the Learning Progress

Let's create a graph to see how well the network learned! This graph shows us how the accuracy improved over time.


In [ ]:
plt.plot(range(1, 6), train_accuracies, 'o-', label='Training Accuracy', linewidth=2, markersize=8)
plt.plot(range(1, 6), val_accuracies, 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.title('How Well the Computer Learned', fontsize=14, fontweight='bold')
plt.xlabel('Epoch (Round of Learning)', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([0, 100])
plt.show()


### 🎮 Step 1.6: Try It Yourself!

Now let's test the computer on some new pictures it's never seen before!

**Use the slider below** to pick a test image (0-99) and see:
- What the computer **thinks** the number is (its prediction)
- What the number **actually** is (the true label)
- How confident the computer is in its guess

Try different numbers and see how well the computer does! 🎯


In [ ]:
index_slider = widgets.IntSlider(
    min=0, 
    max=99, 
    step=1, 
    value=0, 
    description="Test Image #",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
display(index_slider)


In [ ]:
def show_prediction(change=None):
    idx = index_slider.value
    img = test_images[idx]
    true_label = test_labels[idx].item()
    
    model1.eval()
    with torch.no_grad():
        output = model1(img.unsqueeze(0))
        probabilities = torch.softmax(output, dim=1)
        predicted_label = torch.argmax(output, dim=1).item()
        confidence = probabilities[0][predicted_label].item() * 100
    
    plt.figure(figsize=(8, 4))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img.squeeze().numpy(), cmap='gray')
    plt.axis('off')
    plt.title(f'Test Image #{idx}', fontsize=12, fontweight='bold')
    
    plt.subplot(1, 2, 2)
    colors = ['green' if i == predicted_label else 'gray' for i in range(10)]
    plt.bar(range(10), probabilities[0].numpy() * 100, color=colors, alpha=0.7)
    plt.xlabel('Digit', fontsize=11)
    plt.ylabel('Confidence (%)', fontsize=11)
    plt.title(f'Computer\'s Guess: {predicted_label}\n(Confidence: {confidence:.1f}%)', 
              fontsize=12, fontweight='bold')
    plt.xticks(range(10))
    plt.ylim([0, 100])
    plt.grid(True, alpha=0.3, axis='y')
    
    if predicted_label == true_label:
        result_text = f"✅ CORRECT! (True: {true_label})"
        result_color = 'green'
    else:
        result_text = f"❌ WRONG! (True: {true_label})"
        result_color = 'red'
    
    plt.figtext(0.5, 0.02, result_text, ha='center', fontsize=14, 
                fontweight='bold', color=result_color)
    
    plt.tight_layout()
    plt.show()

index_slider.observe(show_prediction, names='value')
show_prediction()


---

## 🧩 Part 2: Too Smart for Its Own Good (Overfitting)

### What's Overfitting?

Imagine you're studying for a test by memorizing every single word in your textbook. You might do great on practice problems from that exact book, but when you get a new test with different questions, you might struggle!

That's called **overfitting** — when a computer "memorizes" the training examples too well and can't handle new situations.

### What We'll Do

We'll compare two models:
- **Small Model:** Simple and straightforward — just a few neurons
- **Big Model:** Very complex with lots of layers and many neurons

Both will learn the same problem, but watch what happens! The big model might do better on the training data but worse on new data.

**Use the dropdown below to switch between models and see the difference!**


### Step 2.1: Creating a Simple Test Problem

For this experiment, we'll use a simpler problem than recognizing numbers. We'll create two groups of points that look like two crescent moons. The computer's job is to learn which points belong to which group.

**Why this problem?** It's easier to visualize and understand than recognizing numbers, but it still shows us how overfitting works!


In [ ]:
X, y = make_moons(noise=0.2, random_state=42, n_samples=1000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


Now we convert this data into a format that PyTorch can work with (called "tensors" — think of them as special arrays of numbers):


In [ ]:
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)


### Step 2.2: Building Two Different Models

Now we'll create two different neural networks:

**Small Model:**
- Just 1 hidden layer with 8 neurons
- Simple and straightforward
- Like a student who learns the main concepts

**Big Model:**
- 5 hidden layers, each with 32 neurons
- Very complex and powerful
- Like a student who tries to memorize every detail

**What is Sigmoid?**

Sigmoid is another activation function (like ReLU, but different). It takes any number and squishes it into a range between 0 and 1. This is perfect for yes/no questions or when we need probabilities. Think of it like a dimmer switch that can only go from "off" (0) to "on" (1), with everything in between.

We use sigmoid in the output layer here because we're answering a yes/no question: "Does this point belong to group 1 or group 2?"


In [ ]:
class SmallModel(nn.Module):
    def __init__(self):
        super(SmallModel, self).__init__()
        self.layer1 = nn.Linear(2, 8)
        self.output = nn.Linear(8, 1)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.sigmoid(self.output(x))
        return x


In [ ]:
class BigModel(nn.Module):
    def __init__(self):
        super(BigModel, self).__init__()
        self.layer1 = nn.Linear(2, 32)
        self.layer2 = nn.Linear(32, 32)
        self.layer3 = nn.Linear(32, 32)
        self.layer4 = nn.Linear(32, 32)
        self.layer5 = nn.Linear(32, 32)
        self.output = nn.Linear(32, 1)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        x = torch.relu(self.layer4(x))
        x = torch.relu(self.layer5(x))
        x = torch.sigmoid(self.output(x))
        return x


### Step 2.3: Training and Comparing the Models

Now we'll create a function that trains whichever model you choose and shows you the results. Watch carefully at the difference between training accuracy and test accuracy!

**What to look for:**
- Does the big model get higher training accuracy? (Probably yes!)
- Does it also get higher test accuracy? (Maybe not!)
- Is there a bigger gap between training and test accuracy for the big model? (This would indicate overfitting!)


In [ ]:
def train_and_plot(size="small"):
    clear_output(wait=True)
    
    if size == "small":
        model = SmallModel()
    else:
        model = BigModel()
    
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()
    
    train_accs = []
    test_accs = []
    
    for epoch in range(20):
        model.train()
        optimizer.zero_grad()
        train_outputs = model(X_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_loss.backward()
        optimizer.step()
        
        train_preds = (train_outputs > 0.5).float()
        train_acc = (train_preds == y_train_tensor).float().mean().item() * 100
        train_accs.append(train_acc)
        
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            test_preds = (test_outputs > 0.5).float()
            test_acc = (test_preds == y_test_tensor).float().mean().item() * 100
            test_accs.append(test_acc)
    
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, 21), train_accs, 'o-', label='Training Accuracy', 
             linewidth=2, markersize=6, color='blue')
    plt.plot(range(1, 21), test_accs, 's-', label='Test Accuracy', 
             linewidth=2, markersize=6, color='orange')
    plt.title(f'{size.upper()} Model: How Well It Learned', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch (Round of Learning)', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 100])
    plt.tight_layout()
    plt.show()
    
    gap = train_accs[-1] - test_accs[-1]
    if gap > 5:
        print(f"⚠️  WARNING: Big gap! Training: {train_accs[-1]:.1f}%, Test: {test_accs[-1]:.1f}%")
        print("   This means the model might be OVERFITTING (memorizing too much).")
    else:
        print(f"✅ Good! Training: {train_accs[-1]:.1f}%, Test: {test_accs[-1]:.1f}%")
        print("   The model is learning general patterns, not just memorizing.")


In [ ]:
dropdown = widgets.Dropdown(
    options=['small', 'big'], 
    value='small', 
    description='Model Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

widgets.interactive(train_and_plot, size=dropdown)


### 🤔 What Did You Notice?

**Think about these questions:**
1. Did the big model get a higher training accuracy than the small model?
2. Did the big model also get a higher test accuracy?
3. What happened to the gap between training and test accuracy?

**Key Insight:** Sometimes being "too smart" (too complex) can actually make you worse at handling new situations! This is why we need to find the "just right" size for our models.


---

## 🧩 Part 3: Draw the Decision Boundary

### What's a Decision Boundary?

When a neural network learns to separate two groups, it draws an invisible line (or curve) between them. This is called a **decision boundary**.

Think of it like drawing a line on the playground to separate two teams. The computer learns where to draw this line to best separate the groups.

### What We'll Do

We'll watch the computer learn to separate two groups of points. As we change the number of neurons, we'll see how the decision boundary changes:
- **Few neurons** = Simple, smooth boundary
- **Many neurons** = Complex, wiggly boundary

**Use the slider below to change the number of neurons and watch the boundary change!**


### Step 3.1: Creating the Data

We'll create a new dataset with two groups of points that look like crescent moons. This time we'll use fewer points so we can see the boundary more clearly.


In [ ]:
X, y = make_moons(noise=0.25, random_state=1, n_samples=300)

X_tensor = torch.FloatTensor(X)
y_tensor = torch.FloatTensor(y).unsqueeze(1)


### Step 3.2: Visualizing the Decision Boundary

Now we'll create a function that:
1. Trains a model with a specific number of neurons
2. Creates a grid covering the entire area
3. Asks the model what it thinks about every point in that grid
4. Colors the background based on the model's predictions
5. Shows the actual data points on top

This lets us "see" the decision boundary that the model learned!

**What is Tanh?**

Tanh (hyperbolic tangent) is another activation function. It's similar to sigmoid but can output values between -1 and 1 (instead of 0 and 1). We use it here because it often creates smoother, more natural-looking decision boundaries for visualization.


In [ ]:
def plot_boundary(neurons=4):
    clear_output(wait=True)
    
    class BoundaryModel(nn.Module):
        def __init__(self, n_neurons):
            super(BoundaryModel, self).__init__()
            self.layer1 = nn.Linear(2, n_neurons)
            self.output = nn.Linear(n_neurons, 1)
        
        def forward(self, x):
            x = torch.tanh(self.layer1(x))
            x = torch.sigmoid(self.output(x))
            return x
    
    model = BoundaryModel(neurons)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()
    
    for epoch in range(30):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    model.eval()
    with torch.no_grad():
        grid_points = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
        Z = model(grid_points).numpy()
        Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, levels=50, alpha=0.6, cmap='coolwarm')
    plt.colorbar(label='Confidence', shrink=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', 
                edgecolors='black', linewidths=1.5, s=50, zorder=3)
    plt.title(f'Decision Boundary with {neurons} Neurons', 
              fontsize=14, fontweight='bold')
    plt.xlabel('X coordinate', fontsize=12)
    plt.ylabel('Y coordinate', fontsize=12)
    
    if neurons <= 4:
        complexity = "Simple, smooth boundary"
    elif neurons <= 16:
        complexity = "Moderately complex boundary"
    else:
        complexity = "Complex, wiggly boundary"
    
    plt.figtext(0.5, 0.02, f'💡 {complexity}', 
                ha='center', fontsize=11, style='italic')
    
    plt.tight_layout()
    plt.show()
    
    with torch.no_grad():
        predictions = (model(X_tensor) > 0.5).float()
        accuracy = (predictions == y_tensor).float().mean().item() * 100
    print(f"Model accuracy: {accuracy:.1f}%")


Now use the slider to change the number of neurons and watch how the boundary changes:


In [ ]:
slider = widgets.IntSlider(
    min=2, 
    max=32, 
    step=2, 
    value=4, 
    description='Number of Neurons:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

widgets.interactive(plot_boundary, neurons=slider)


### 🎨 What Did You See?

**Try moving the slider and observe:**
- With **few neurons** (2-4): The boundary is simple and smooth, like a gentle curve
- With **more neurons** (16-32): The boundary becomes wiggly and complex, trying to fit every point perfectly

**Think about it:**
- Which boundary do you think would work better on NEW data?
- Is a wiggly boundary always better, or can it be "too wiggly"?

**Key Insight:** More neurons give the model more flexibility, but sometimes simpler is better! A smooth boundary often works better on new data than a super wiggly one.


---

## 🧩 Wrap-Up & Reflection

Great job exploring neural networks! Let's think about what we learned.

### Questions to Discuss (Write your answers or discuss with a partner):

1. **Part 1 - Reading Numbers:**
   - What did the model learn to do?
   - Why do you think it got better over time?
   - Did it get every test image correct? Why or why not?

2. **Part 2 - Too Smart for Its Own Good:**
   - What happened when you compared the small and big models?
   - Why did the big model sometimes do worse on new data?
   - What does this teach us about learning in general?

3. **Part 3 - Decision Boundaries:**
   - What happened to the boundary shape when you increased the number of neurons?
   - Do you think a wiggly boundary is always better? Why or why not?
   - What's the "just right" number of neurons?

### ⭐ Key Ideas to Remember:

1. **Neural networks learn patterns** by looking at lots of examples, just like we do!

2. **Size matters!** 
   - Too small = can't learn complex patterns
   - Too big = memorizes instead of learning general patterns
   - Just right = learns patterns that work on new data

3. **The goal is generalization** — we want models that work well on NEW data, not just the data they trained on.

### 🎓 What's Next?

Now that you understand the basics, you could:
- Try changing the number of epochs (training rounds) and see what happens
- Experiment with different numbers of neurons in Part 1
- Create your own classification problem!

**Congratulations on completing the Neural Network Playground! 🎉**

---

## 📋 Submission Requirements

To complete this lab, submit the following:

### Screenshots (2 required):
1. **Small Model Results** - Screenshot of the training and test accuracy graph from Part 2 when using the "small" model
2. **Big Model Results** - Screenshot of the training and test accuracy graph from Part 2 when using the "big" model (showing the gap between training and test accuracy)

### Text Response:
Answer the three reflection questions above:
1. **Part 1 - Reading Numbers:** What did the model learn to do? Why do you think it got better over time? Did it get every test image correct? Why or why not?
2. **Part 2 - Too Smart for Its Own Good:** What happened when you compared the small and big models? Why did the big model sometimes do worse on new data? What does this teach us about learning in general?
3. **Part 3 - Decision Boundaries:** What happened to the boundary shape when you increased the number of neurons? Do you think a wiggly boundary is always better? Why or why not? What's the "just right" number of neurons?
